In [ ]:
using Statistics
using CairoMakie
using CairoMakie.Colors
using Oceananigans
using Interpolations
include("src-fig/plotting.jl")
foldername = "../scratch/turbulence-at-fronts/Strain"

DFM = joinpath(foldername, "output.jld2")
iterations, times = iterations_times(DFM)
sp = simulation_parameters(DFM)
xsᶜ, xsᶠ, ysᶜ, ysᶠ, zsᶜ, zsᶠ = grid_nodes(DFM)
inds = center_indices(DFM)
frames = [401, 581, 941];

In [ ]:
MBALANCE = joinpath(foldername, "MBALANCE.jld2")
jldopen(file->keys(file["timeseries"]), MBALANCE)
i1, i2 = zᶜbounds(DFM, -20, -4)
avg_iterations = iterations[1001:1201]

DM²Dt = timeseries_of(x->mean(x[:, i1:i2]) ./ sp.α, MBALANCE, "DM²Dt", iterations)
M² = timeseries_of(x->mean(x[:, i1:i2]), MBALANCE, "M²", iterations)
∂u∂xM² = -timeseries_of(x->mean(x[:, i1:i2]) ./ sp.α, MBALANCE, "∂u∂xM²", iterations)
∂w∂xN² = -timeseries_of(x->mean(x[:, i1:i2]) ./ sp.α, MBALANCE, "∂w∂xN²", iterations)
F_M² = timeseries_of(x->mean(x[:, i1:i2]) ./ sp.α, MBALANCE, "F", iterations)

DM²Dt_avg = time_average_of(x->x ./ sp.α, MBALANCE, "DM²Dt", avg_iterations)
M²_avg = time_average_of(x->x, MBALANCE, "M²", avg_iterations)
∂u∂xM²_avg = -time_average_of(x->x ./ sp.α, MBALANCE, "∂u∂xM²", avg_iterations)
∂w∂xN²_avg = -time_average_of(x->x ./ sp.α, MBALANCE, "∂w∂xN²", avg_iterations)
F_M²_avg = time_average_of(x->x ./ sp.α, MBALANCE, "F", avg_iterations)
b_avg = time_average_of(x->x, DFM, "b_dfm", avg_iterations)

αM² = M²;
αM²_avg = M²_avg

NBALANCE = joinpath(foldername, "NBALANCE.jld2")
jldopen(file->keys(file["timeseries"]), NBALANCE)
i1, i2 = zᶜbounds(DFM, -20, -2)

DN²Dt = timeseries_of(x->mean(x[:, i1:i2]) ./ sp.α, NBALANCE, "DN²Dt", iterations)
N² = timeseries_of(x->mean(x[:, i1:i2]), NBALANCE, "N²", iterations)
∂u∂zM² = -timeseries_of(x->mean(x[:, i1:i2]) ./ sp.α, NBALANCE, "∂u∂zM²", iterations)
∂w∂zN² = -timeseries_of(x->mean(x[:, i1:i2]) ./ sp.α, NBALANCE, "∂w∂zN²", iterations)
F_N² = timeseries_of(x->mean(x[:, i1:i2]) ./ sp.α, NBALANCE, "F", iterations)

DN²Dt_avg = time_average_of(x->x ./ sp.α, NBALANCE, "DN²Dt", avg_iterations)
N²_avg = time_average_of(x->x, NBALANCE, "N²", avg_iterations)
∂u∂zM²_avg = -time_average_of(x->x ./ sp.α, NBALANCE, "∂u∂zM²", avg_iterations)
∂w∂zN²_avg = -time_average_of(x->x ./ sp.α, NBALANCE, "∂w∂zN²", avg_iterations)
F_N²_avg = time_average_of(x->x ./ sp.α, NBALANCE, "F", avg_iterations);

In [ ]:
fig = Figure()
ax = Axis(fig[1, 1];
    title = L"\text{Horizontal and surface average}",
    xlabel = L"t / \text{hr}",
    ylabel = L"A / \alpha \, \text{s}^{-2}",
    limits = (0, 200, -1e-6, 1e-6)
)

ln1 = lines!(ax, times ./ 3600, filt(DM²Dt, 1); color=:black)
lines!(ax, times ./ 3600, filt(∂w∂xN² .+ ∂u∂xM² .+ F_M² .+ αM², 1); color=:gray)
ln2 = lines!(ax, times ./ 3600, filt(αM², 1))
ln3 = lines!(ax, times ./ 3600, filt(∂u∂xM², 1))
ln4 = lines!(ax, times ./ 3600, filt(∂w∂xN², 1))
ln5 = lines!(ax, times ./ 3600, filt(F_M², 1))
lns = [ln1, ln2, ln3, ln4, ln5]
labels = [L"DM^2 / Dt", L"\alpha M^2", L"-\partial u/\partial x M^2", L"-\partial w/\partial x N^2", L"F_{M^2}"]

Legend(fig[1, 2], lns, labels)
fig

In [ ]:
fig = Figure()
ax = Axis(fig[1, 1];
    title = L"\text{Horizontal and surface average}",
    xlabel = L"t / \text{hr}",
    ylabel = L"A / \alpha \, \text{s}^{-2}",
    limits = (0, 200, -3e-5, 3e-5)
)

ln1 = lines!(ax, times ./ 3600, filt(DN²Dt, 1); color=:black)
lines!(ax, times ./ 3600, filt(∂w∂zN² .+ ∂u∂zM² .+ F_N², 1); color=:gray)
ln3 = lines!(ax, times ./ 3600, filt(∂u∂zM², 1))
ln4 = lines!(ax, times ./ 3600, filt(∂w∂zN², 1))
ln5 = lines!(ax, times ./ 3600, filt(F_N², 1))
lns = [ln1, ln3, ln4, ln5]
labels = [L"DN^2 / Dt", L"-\partial u/\partial z M^2", L"-\partial w/\partial z N^2", L"F_{N^2}"]

Legend(fig[1, 2], lns, labels)
fig

In [ ]:
fig = Figure()
ax = Axis(fig[1, 1];
    title = L"\text{Final state, average over } [-20\,\text{m}, -4\,\text{m}]",
    xlabel = L"x / \text{km}",
    ylabel = L"A / \alpha \, \text{s}^{-2}",
    limits = (-2, -1.5, -1.2e-5, 1.2e-5)
)

ln1 = lines!(ax, xsᶠ./ 1000, filt(mean(DM²Dt_avg[:, i1:i2]; dims=2)[:, 1], 3); color=:black)
ln2 = lines!(ax, xsᶠ./ 1000, filt(mean(αM²_avg[:, i1:i2]; dims=2)[:, 1], 3))
ln3 = lines!(ax, xsᶠ./ 1000, filt(mean(∂u∂xM²_avg[:, i1:i2]; dims=2)[:, 1], 3))
ln4 = lines!(ax, xsᶠ./ 1000, filt(mean(∂w∂xN²_avg[:, i1:i2]; dims=2)[:, 1], 3))
ln5 = lines!(ax, xsᶠ./ 1000, filt(mean(F_M²_avg[:, i1:i2]; dims=2)[:, 1], 3))
lns = [ln1, ln2, ln3, ln4, ln5]
labels = [L"DM^2 / Dt", L"\alpha M^2", L"-\partial u/\partial x M^2", L"-\partial w/\partial x N^2", L"F_{M^2}"]

Legend(fig[1, 2], lns, labels)

ax = Axis(fig[1, 1],
    limits = (-2, -1.5, nothing, nothing)
)
hidexdecorations!(ax)
hideydecorations!(ax)
lines!(ax, xsᶜ ./ 1000, filt(mean(b_avg[:, i1:i2]; dims=2)[:, 1], 3); color=:black, linestyle=:dash)
fig

In [ ]:
fig = Figure()
ax = Axis(fig[1, 1];
    title = L"\text{Final state, average over } [-20\,\text{m}, -4\,\text{m}]",
    xlabel = L"x / \text{km}",
    ylabel = L"A / \alpha \, \text{s}^{-2}",
    limits = (-2, -1.5, -3e-4, 3e-4)
)

ln1 = lines!(ax, xsᶜ./ 1000, filt(mean(DN²Dt_avg[:, i1:i2]; dims=2)[:, 1], 3); color=:black)
ln3 = lines!(ax, xsᶜ./ 1000, filt(mean(∂u∂zM²_avg[:, i1:i2]; dims=2)[:, 1], 3))
ln4 = lines!(ax, xsᶜ./ 1000, filt(mean(∂w∂zN²_avg[:, i1:i2]; dims=2)[:, 1], 3))
ln5 = lines!(ax, xsᶜ./ 1000, filt(mean(F_N²_avg[:, i1:i2]; dims=2)[:, 1], 3))
lns = [ln1, ln3, ln4, ln5]
labels = [L"DN^2 / Dt", L"-\partial u/\partial z M^2", L"-\partial w/\partial z N^2", L"F_{N^2}"]

Legend(fig[1, 2], lns, labels)

ax = Axis(fig[1, 1],
    limits = (-2, -1.5, nothing, nothing)
)
hidexdecorations!(ax)
hideydecorations!(ax)
ln0 = lines!(ax, xsᶜ ./ 1000, filt(mean(b_avg[:, i1:i2]; dims=2)[:, 1], 3); color=:black, linestyle=:dash)
fig

In [ ]:
limits = (-2.5, 0.5, -50, 0)
colorrange = (-3, 3)
colormap = :balance
levels = 20
σ = (3, 1)
M²_max = maximum(abs, M²_avg)

fig = Figure(; size=(900, 800), fontsize=18)
ax = Axis(fig[1, 1]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"DM^2 / Dt \text{ (actual)}"
)
hidexdecorations!(ax; ticks=false)
#hideydecorations!(ax; ticks=false)
heatmap!(ax, xsᶠ ./ 1000, zsᶜ, filt(DM²Dt_avg, σ) ./ M²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)

ax = Axis(fig[1, 2]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"DM^2 / Dt \text{ (sum)}"
)

hidexdecorations!(ax; ticks=false)
hideydecorations!(ax; ticks=false)
heatmap!(ax, xsᶠ ./ 1000, zsᶜ, filt(∂w∂xN²_avg .+ ∂u∂xM²_avg .+ F_M²_avg .+ αM²_avg, σ) ./ M²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)

ax = Axis(fig[2, 1]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"-\partial w / \partial x N^2"
)
hidexdecorations!(ax; ticks=false)
#hideydecorations!(ax; ticks=false)
heatmap!(ax, xsᶠ ./ 1000, zsᶜ, filt(∂w∂xN²_avg, σ) ./ M²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)

ax = Axis(fig[2, 2]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"-\partial u / \partial x M^2-\partial w / \partial x N^2"
)

hidexdecorations!(ax; ticks=false)
hideydecorations!(ax; ticks=false)
heatmap!(ax, xsᶠ ./ 1000, zsᶜ, filt(∂u∂xM²_avg .+ ∂w∂xN²_avg , σ) ./ M²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)


ax = Axis(fig[3, 1]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"\alpha M^2"
)
#hidexdecorations!(ax; ticks=false)
#hideydecorations!(ax; ticks=false)
heatmap!(ax, xsᶠ ./ 1000, zsᶜ, 10filt(αM²_avg, σ) ./ M²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)

ax = Axis(fig[3, 2]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"F_{M^2}"
)

#hidexdecorations!(ax; ticks=false)
hideydecorations!(ax; ticks=false)
ht = heatmap!(ax, xsᶠ ./ 1000, zsᶜ, filt(F_M²_avg .+ 0∂w∂xN²_avg , σ) ./ M²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)
Colorbar(fig[1:3, 3], ht; label=L"A/\alpha M²_\text{max}")

function sc_vec(x, z)
    Point2(u_interp(x, z), w_interp(x, z))
end

sc_xs = xsᶜ[inds[1:20:end]]
sc_zs = zsᶜ[1:10:end]
sc_us = [sc_vec(x, z)[1] for x in sc_xs, z in sc_zs] ./ 1000
sc_vs = [sc_vec(x, z)[2] for x in sc_xs, z in sc_zs]

using LinearAlgebra
sc_color = vec(sqrt.(0sc_us.^2 .+ sc_vs.^2))

arrows2d!(ax, sc_xs ./ 1000, sc_zs, sc_us, sc_vs; colormap=sc_colormap, lengthscale = 4000, color=sc_color)

fig

In [ ]:
limits = (-2.5, 2.5, -100, 0)
colorrange = (-20, 20)
colormap = :balance
levels = 20
σ = (10, 0)
N²_max = maximum(abs, N²_avg[inds, end-50:end-10])

fig = Figure(; size=(900, 800), fontsize=18)
ax = Axis(fig[1, 1]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"DN^2 / Dt \text{ (actual)}"
)
hidexdecorations!(ax; ticks=false)
#hideydecorations!(ax; ticks=false)
heatmap!(ax, xsᶜ ./ 1000, zsᶠ, filt(DN²Dt_avg, σ) ./ N²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)

ax = Axis(fig[1, 2]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"DN^2 / Dt \text{ (sum)}"
)

hidexdecorations!(ax; ticks=false)
hideydecorations!(ax; ticks=false)
heatmap!(ax, xsᶜ ./ 1000, zsᶠ, filt(∂w∂zN²_avg .+ ∂u∂zM²_avg .+ F_N²_avg, σ) ./ N²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)

ax = Axis(fig[2, 1]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"-\partial w / \partial z N^2"
)
hidexdecorations!(ax; ticks=false)
#hideydecorations!(ax; ticks=false)
heatmap!(ax, xsᶜ ./ 1000, zsᶠ, filt(∂w∂zN²_avg, σ) ./ N²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)

ax = Axis(fig[2, 2]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"-\partial u / \partial z M^2"
)

hidexdecorations!(ax; ticks=false)
hideydecorations!(ax; ticks=false)
heatmap!(ax, xsᶜ ./ 1000, zsᶠ, filt(∂u∂zM²_avg , σ) ./ N²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)


ax = Axis(fig[3, 1]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"N^2"
)
#hidexdecorations!(ax; ticks=false)
#hideydecorations!(ax; ticks=false)
heatmap!(ax, xsᶜ ./ 1000, zsᶠ, filt(N²_avg, σ) ./ N²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)

ax = Axis(fig[3, 2]; 
    limits,
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}", 
    title = L"F_{N^2}"
)

#hidexdecorations!(ax; ticks=false)
hideydecorations!(ax; ticks=false)
ht = heatmap!(ax, xsᶜ ./ 1000, zsᶠ, filt(F_N²_avg .+ 0∂w∂zN²_avg , σ) ./ N²_max; colormap, colorrange)
contour!(ax, xsᶜ ./ 1000, zsᶜ, b_avg; levels, color=:black)

Colorbar(fig[1:3, 3], ht; label=L"A/\alpha N²_\text{max}")

fig

In [ ]:
VBALANCE = joinpath(foldername, "VBALANCE.jld2")
TKE = joinpath(foldername, "TKE.jld2")
PV = joinpath(foldername, "PV.jld2")

DvDt = time_average_of(x->x ./ sp.f, VBALANCE, "DvDt", iterations[1001:1201])
αv = time_average_of(x->sp.α .* x ./ sp.f, DFM, "v_dfm", iterations[1001:1201])
fu = time_average_of(identity, DFM, "u_dfm", iterations[1001:1201])
turbulence_h = time_average_of(x->x ./ sp.f, VBALANCE, "turbulence_h", iterations[1001:1201])
turbulence_z = time_average_of(x->x ./ sp.f, VBALANCE, "turbulence_z", iterations[1001:1201]);

In [ ]:
colorrange = (-0.1, 0.1)
fig = Figure(; size=(800, 300), fontsize=18)
ax = Axis(fig[1, 1]; 
    limits = (-sp.Lh/2000, sp.Lh/2000, -sp.H, 0), 
    title = L"\text{Along front tendency}", 
    xticks = [-2, -1, 0, 1, 2],
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}"
)
heatmap!(ax, xsᶜ ./ 1000, zsᶜ, -fu[1:end-1, :] .- αv .+ turbulence_z .+ turbulence_h; colorrange, colormap=:balance)

ax = Axis(fig[1, 2]; 
    limits = (-sp.Lh/2000, sp.Lh/2000, -sp.H, 0), 
    title = L"\text{No turbulence}",
    xticks = [-2, -1, 0, 1, 2],
    xlabel = L"x/\text{km}",
    ylabel = L"z/\text{m}"
)
heatmap!(ax, xsᶜ ./ 1000, zsᶜ, -fu[1:end-1, :] .- αv; colorrange, colormap=:balance)
hideydecorations!(ax; ticks=false, grid=false)

fig

In [ ]:
xs = range(-1.0, 1.2, 1000)
zs = range(-1, 0, 1000)

fig = Figure(; size=(600, 240), fontsize=18)
ax = Axis(fig[1, 3]; xlabel="Across-front", ylabel="Vertical", ylabelsize=16, xlabelsize=16)

arrest_x = -0.5
arrest_z = -0.1

ψσ = 0.1
dx = arrest_x - 0.5
dz = arrest_z - 1.0
dl = sqrt(dx^2 + dz^2)
arrest_n = [dx; dz] ./ dl
arrest_m = [-dz; dx] ./ dl

entrainment_x = arrest_x - dx / 2
entrainment_z = arrest_z + dz / 2

Σ² = arrest_n * (4/ψσ^2) * transpose(arrest_n) + arrest_m * (40/dl^2) * transpose(arrest_m)

# Arrest region streamfunction 
function arrest_ψ(x, z) 
    r = (x-arrest_x)^2 + (z-arrest_z)^2
    -exp(-r / 2ψσ^2) * (1 + 0.2 * sin(100z))
end

arrest_colormap = [RGBA(0, 0, 1, 0), RGBA(0, 0, 1, 1)]

arrest_u(x, z) = -(arrest_ψ(x, z+5e-5) - arrest_ψ(x, z-5e-5)) / 1e-4
arrest_w(x, z) = (arrest_ψ(x+5e-5, z) - arrest_ψ(x-5e-5, z)) / 1e-4

arrest_U(x, z) = Point2(arrest_u(x, z), arrest_w(x, z))

# Entrainment region streamfunction
function entrainment_ψ(x, z) 
    r = [x-entrainment_x; z-entrainment_z]
    -exp(-transpose(r) * Σ² * r/ 2)
end
entrainment_colormap = [RGBA(1, 0, 0, 0), RGBA(1, 0, 0, 1)]

function entrainment_u(x, z) 
    u = -(entrainment_ψ(x, z+5e-5) - entrainment_ψ(x, z-5e-5)) / 1e-4
    return u
end
function entrainment_w(x, z) 
    w = (entrainment_ψ(x+5e-5, z) - entrainment_ψ(x-5e-5, z)) / 1e-4
    return w
end

entrainment_U(x, z) = Point2(entrainment_u(x, z), entrainment_w(x, z))

function buoyancy(x, z)
    tanh(3abs(z-1)*(x -(-1+0.5* sqrt(1+(2.5z)^2))))
end

function underneath(x, z) 
    -entrainment_ψ(x+0.1, z)
end

heatmap!(ax, xs, zs, underneath; colormap=entrainment_colormap, colorrange=(0, 2))

contour!(ax, xs, zs, buoyancy; color=:black)
streamplot!(ax, arrest_U, xs, zs; colormap=arrest_colormap, arrow_size=10)
streamplot!(ax, entrainment_U, xs, zs; colormap=entrainment_colormap, arrow_size=10)
gl = GridLayout(fig[1, 3], 
    tellwidth = false, 
    tellheight = false, 
    halign = 0.95, 
    valign = 0.3,
)
Label(gl[1, 1], "Transport of\nthermocline\nPV";
    fontsize=12,
    padding = (1, 1, 2, 2),
    color=:red
)
gl = GridLayout(fig[1, 3], 
    tellwidth = false, 
    tellheight = false, 
    halign = 0.95, 
    valign = 1.0,
)
Label(gl[1, 1], "Vigourous mixing of\nnegative and\npositive PV fluid\ndriven by\nsurface cooling";
    fontsize=12,
    padding = (1, 1, 2, 2),
    color=:blue
)

gl = GridLayout(fig[1, 3], 
    tellwidth = false, 
    tellheight = false, 
    halign = 0.05, 
    valign = 0.1,
)
Label(gl[1, 1], "Stable fluid\nejected\nunder front";
    fontsize=12,
    padding = (1, 1, 2, 2),
    color=:red
)
hidexdecorations!(ax; label=false)
hideydecorations!(ax; label=false)



q = time_average_of(joinpath(foldername, "PV.jld2"), "q_dfm", iterations[901:1001]) do field
    filt(field, 0, 0) ./ (sp.f * sp.N₀²)
end

uq = time_average_of(joinpath(foldername, "PV.jld2"), "uq_dfm", iterations[901:1001]) do field
    filt(field, 0, 0) ./ (sp.f * sp.N₀²)
end
wq = time_average_of(joinpath(foldername, "PV.jld2"), "wq_dfm", iterations[901:1001]) do field
    filt(field, 0, 0) ./ (sp.f * sp.N₀²)
end

b = time_average_of(a->a, joinpath(foldername, "DFM.jld2"), "b_dfm", iterations[911:1001])
# Extremely smoothed versions of the secondary circulation?
u = time_average_of(a->a, joinpath(foldername, "DFM.jld2"), "u_dfm", iterations[911:1001]) .+ [-sp.α * x for x in xsᶠ, z in 1:1]
w = time_average_of(a->a, joinpath(foldername, "DFM.jld2"), "w_dfm", iterations[911:1001])

# Interpolate u and w
u_interp = linear_interpolation((xsᶠ, zsᶜ), u; extrapolation_bc=Linear())
w_interp = linear_interpolation((xsᶜ, zsᶠ), w; extrapolation_bc=Linear())

function sc_vec(x, z)
    2e4 .* Point2(u_interp(sp.Lh * x, sp.H * z) / sp.Lh, w_interp(sp.Lh * x, sp.H * z) / sp.H)
end


colormap = to_colormap(:balance)

ax = Axis(fig[1, 1]; xlabel=L"x / \text{km}", ylabel=L"z / \text{m}", limits=(-sp.Lh / 2000, sp.Lh / 2000, -sp.H, 0), xticks=[-2, -1, 0, 1, 2])
ht = heatmap!(ax, xsᶠ ./ 1000, zsᶜ, q; colormap, colorrange=(-0.3, 0.3), highclip=colormap[end], lowclip=colormap[1])
contour!(ax, xsᶜ ./ 1000, zsᶜ, b; color=(:black, 0.5), levels=b_levels)

ax = Axis(fig[1, 1]; limits=(-0.5, 0.5, -1, 0))
hidexdecorations!(ax)
hideydecorations!(ax)

sc_colormap = [RGBA(0, 0, 0, 0), RGBA(0, 0, 0, 1)]

sc_xs = range(-0.5, 0.5, 6)
sc_zs = range(-0.8, -0.1, 6)
sc_us = [sc_vec(x, z)[1] for x in sc_xs, z in sc_zs]
sc_vs = [sc_vec(x, z)[2] for x in sc_xs, z in sc_zs]

using LinearAlgebra
sc_color = sqrt.(sc_us.^2 .+ sc_vs.^2)
sc_us = sc_us ./ sc_color
sc_vs = sc_vs ./ sc_color
sc_color = vec(sqrt.(0sc_us.^2 .+ sc_vs.^2))

arrows2d!(ax, sc_xs, sc_zs, sc_us, sc_vs; colormap=sc_colormap, lengthscale = 0.1, color=sc_color)

Colorbar(fig[1, 2], ht; label=L"\overline{q} / fN_0^2")

subfig_label!(fig[1, 1], 1)
subfig_label!(fig[1, 3], 2)

save("figures/paper/cartoon.png", fig; px_per_unit=2)

fig